In [3]:
"""
재료명 정규화 및 ingredients 테이블 재구축 스크립트

1. recipe_ingredients.name 전체 추출
2. 더미/조리메모 제거
3. 정규화 (공백/약/과/줄 등 제거, 중복 통합)
4. ingredients 테이블 재구축
5. recipe_ingredients.ingredient_id 재매핑
"""

import re
import mysql.connector
from mysql.connector import Error
import logging

# ─────────────────────────────────────────────
# 설정값
# ─────────────────────────────────────────────
DB_CONFIG = {
    "host": "localhost",
    "port": 3306,
    "database": "cooking_db",
    "user": "root",
    "password": "root",
    "charset": "utf8mb4"
}

# ─────────────────────────────────────────────
# 로깅
# ─────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler("normalize_ingredients.log", encoding="utf-8")
    ]
)
log = logging.getLogger(__name__)


# ─────────────────────────────────────────────
# 더미 판별 패턴
# ─────────────────────────────────────────────
DUMMY_PATTERNS = [
    r'재료$', r'만드는\s*법', r'버전$', r'법$', r'세팅', r'인분',
    r'목록', r'쌀뜨물', r'준비', r'분량', r'기준', r'응용',
    r'업그레이드', r'도시락', r'볼에', r'냄비에', r'팬에',
    r'추가재료', r'기타', r'삼등분', r'두께', r'가로',
    r'만드는법', r'손질법', r'아셨쥬', r'느끼해', r'비슷하고',
    r'없죠', r'전량$', r'인분$', r'세팅$', r'채 썰어',
    r'볼에 간', r'볼에 청', r'소스 볼', r'팬에 남은',
    r'완전 쉬운', r'명절 앞둔', r'겨자소스 비율', r'겨자소스 만드는',
    r'감자탕 만드는', r'겨자소스 비율', r'느끼해지려고',
    r'는 알리오', r'도시락까지', r'해보니까',
]

DUMMY_EXACT = {
    '목록', '쌀뜨물', '정수물', '주재료', '주 재료', '준비', '준비하기',
    '준비 법', '양념', '쌈장', '세팅', '추가재료', '기타 재료',
    '기타', '블럭', '종이컵', '마리', '포기', '기본버전', '간단버전',
    '한판버전', '달걀물버전', '간장버전', '고추냉이버전', '프라이팬 버전',
    '에어프라이어 버전', '튀긴 두부 버전', '으깬 두부 버전', '우유 버전',
    '기본 버전', '오리지널 버전 만드는 법', '업그레이드 버전 만드는 법',
    '게살버전', '참치버전', '고추장버전',
}

def is_dummy(name: str) -> bool:
    """더미/조리메모 판별"""
    n = name.strip()
    if n in DUMMY_EXACT:
        return True
    if len(n) <= 1:
        return True
    if len(n) >= 20:  # 너무 긴 건 문장
        return True
    for pat in DUMMY_PATTERNS:
        if re.search(pat, n):
            return True
    return False


# ─────────────────────────────────────────────
# 재료명 정규화
# ─────────────────────────────────────────────

# 수량/단위 표현 제거
UNIT_SUFFIXES = re.compile(
    r'\s*(약\s*과|약\s*와|약과|약와|약\s*대|약\s*줄|약\s*장|약\s*캔|약\s*봉|약\s*모|약\s*통|약\s*마리|약\s*알|약\s*개|'
    r'약$|과$|와$|줄$|팩$|캔$|장$|봉지$|봉$|모$|통$|마리$|대$|줌$|조각$|알$|포$|인분$|컵$|개$|'
    r'\d+[\w가-힣]*$)'
)

# 공백 통일 (연속 공백 → 단일)
def normalize_spaces(name: str) -> str:
    return re.sub(r'\s+', ' ', name).strip()

# 동의어 매핑 (정규화된 이름 → 대표 이름)
SYNONYM_MAP = {
    '계란':     '달걀',
    '달걀 개':  '달걀',
    '간마늘':   '다진마늘',
    '간 마늘':  '다진마늘',
    '다진 마늘':'다진마늘',
    '간생강':   '생강',
    '간 생강':  '생강',
    '고운고춧가루': '고운 고춧가루',
    '굵은고춧가루': '굵은 고춧가루',
    '고춧가루':  '굵은 고춧가루',
    '진간장':   '간장',
    '국간장':   '간장',
    '소금':     '꽃소금',
    '꽃소금과 후춧가루': None,  # None이면 분리 불가 → 스킵
    '파프리카 각 씩 청': None,
    '모짜렐라치즈': '모짜렐라 치즈',
    '모차렐라치즈': '모짜렐라 치즈',
    '모차렐라 치즈': '모짜렐라 치즈',
    '통깨':     '참깨',
    '깨소금':   '참깨',
    '간깨':     '참깨',
    '갈은 깨':  '참깨',
    '느타리 버섯': '느타리버섯',
    '새송이버섯': '새송이',
    '양송이 버섯': '양송이버섯',
    '굴 소스':  '굴소스',
    '올리브유':  '올리브오일',
    '올리브오일': '올리브오일',
    '맛술':     '미림',
    '미림':     '미림',
    '후추가루':  '후추',
    '후춧가루':  '후추',
    '통후춧가루': '후추',
    '황설탕':   '설탕',
    '뉴슈가':   '설탕',
    # 스킵할 것들
    '건더기 스프': None,
    '라면 봉지 분말 스프': None,
    '분말스프':  None,
    '정수물':   None,
    '정수 물':  None,

    # 실수로 None 처리됐던 진짜 재료들 복구
    '간장':     '간장',
    '사과':     '사과',
    '닭봉':     '닭봉',
    '닭 마리':  '닭',
    '김 장':    '김',
    '김 팩':    '김',
    '김밥김 장': '김',

    # 미매핑 중 진짜 재료들 추가
    '닭가슴살 팩': '닭가슴살',
    '목살':     '돼지 목살',
    '돼지목살':  '돼지 목살',
    '대패삼겹살 팩': '대패삼겹살',
    '스팸':     '스팸',
    '런천미트':  '스팸',
    '슬라이스 햄': '햄',
    '슬라이스햄 장': '햄',
    '베이컨 장':  '베이컨',
    '베이컨 줄':  '베이컨',
    '소시지':   '소시지',
    '비엔나소시지': '소시지',
    '옛날소시지': '소시지',
    '프랑크소시지': '소시지',
    '분홍소시지': '소시지',
    '어묵 장':  '어묵',
    '사각어묵 장': '어묵',
    '모둠어묵':  '어묵',
    '봉어묵':   '어묵',
    '얇은 사각어묵 장': '어묵',
    '두부 모':  '두부',
    '두부 팩':  '두부',
    '두부반모':  '두부',
    '순두부 팩': '순두부',
    '깍두기':   '깍두기',
    '막김치':   '김치',
    '배추김치':  '김치',
    '신김치':   '김치',
    '자른 김치': '김치',
    '양파 간 것': '양파',
    '껍질양파 약': '양파',
    '쪽파 줄':  '쪽파',
    '쪽파 뿌리': '쪽파',
    '대파 대':  '대파',
    '대파 뿌리': '대파',
    '대파 파란 부분': '대파',
    '팽이버섯':  '팽이버섯',
    '표고버섯':  '표고버섯',
    '표고버섯 약': '표고버섯',
    '표고버섯 과': '표고버섯',
    '당면':     '당면',
    '건당면':   '당면',
    '불린 당면': '당면',
    '불린당면':  '당면',
    '소면':     '소면',
    '건소면':   '소면',
    '칼국수 면': '칼국수면',
    '우동면':   '우동면',
    '우동사리':  '우동면',
    '스파게티 면': '스파게티면',
    '스파게티면': '스파게티면',
    '파스타면':  '스파게티면',
    '라면사리':  '라면',
    '라면 봉지': '라면',
    '밥 공기':  '밥',
    '즉석밥':   '밥',
    '공깃밥':   '밥',

    # 미매핑 중 진짜 재료 추가
    '된장':     '된장',
    '순대':     '순대',
    '춘장':     '춘장',
    '효모':     '효모',
    '쌀 약':    '쌀',
    '밥 약':    '밥',
    '무 과':    '무',
    '무 약':    '무',
}

def normalize_name(raw: str) -> str | None:
    """재료명 정규화 → 대표 이름 반환 (None이면 스킵)"""
    name = normalize_spaces(raw)

    # 동의어 매핑 먼저
    if name in SYNONYM_MAP:
        return SYNONYM_MAP[name]

    # 수량 접미사 제거
    name = UNIT_SUFFIXES.sub('', name).strip()
    name = normalize_spaces(name)

    # 정제 후 재확인
    if name in SYNONYM_MAP:
        return SYNONYM_MAP[name]

    # 더미 재확인
    if is_dummy(name):
        return None

    if len(name) <= 1:
        return None

    return name


# ─────────────────────────────────────────────
# 메인 실행
# ─────────────────────────────────────────────
def main():
    log.info("=== 재료 정규화 시작 ===")

    try:
        conn = mysql.connector.connect(**DB_CONFIG)
        cursor = conn.cursor(dictionary=True)
        log.info("DB 연결 성공")
    except Error as e:
        log.error(f"DB 연결 실패: {e}")
        return

    try:
        # ── 백업
        log.info("백업 테이블 생성 중...")
        cursor.execute("DROP TABLE IF EXISTS ingredients_backup")
        cursor.execute("CREATE TABLE ingredients_backup AS SELECT * FROM ingredients")
        cursor.execute("DROP TABLE IF EXISTS recipe_ingredients_backup")
        cursor.execute("CREATE TABLE recipe_ingredients_backup AS SELECT * FROM recipe_ingredients")
        conn.commit()
        log.info("백업 완료 (ingredients_backup / recipe_ingredients_backup)")

        # ── recipe_ingredients.name 전체 추출
        cursor.execute("SELECT DISTINCT name FROM recipe_ingredients WHERE name IS NOT NULL")
        all_names = [row['name'] for row in cursor.fetchall()]
        log.info(f"고유 재료명 {len(all_names)}개 추출")

        # ── 정규화
        normalized = {}  # 원본명 → 정규화명
        valid_names = set()

        for raw in all_names:
            norm = normalize_name(raw)
            normalized[raw] = norm
            if norm:
                valid_names.add(norm)

        skipped = [k for k, v in normalized.items() if v is None]
        log.info(f"유효 재료: {len(valid_names)}개 / 스킵: {len(skipped)}개")
        log.info(f"스킵된 재료 샘플: {skipped[:20]}")

        # ── ingredients 테이블 초기화 후 재구축
        log.info("ingredients 테이블 재구축 중...")

        # FK 체크 잠시 끄기
        cursor.execute("SET FOREIGN_KEY_CHECKS = 0")

        # recipe_ingredients ingredient_id 전체 NULL로 초기화
        cursor.execute("UPDATE recipe_ingredients SET ingredient_id = NULL")

        # ingredients 전체 삭제
        cursor.execute("DELETE FROM ingredients")

        # AUTO_INCREMENT 리셋
        cursor.execute("ALTER TABLE ingredients AUTO_INCREMENT = 1")

        conn.commit()

        # 유효 재료명 삽입
        name_to_id = {}
        for name in sorted(valid_names):
            cursor.execute(
                "INSERT INTO ingredients (name) VALUES (%s)",
                (name,)
            )
            cursor.execute("SELECT LAST_INSERT_ID() as id")
            new_id = cursor.fetchone()['id']
            name_to_id[name] = new_id

        conn.commit()
        log.info(f"ingredients 테이블에 {len(name_to_id)}개 재료 삽입 완료")

        # ── recipe_ingredients.ingredient_id 재매핑
        log.info("recipe_ingredients 재매핑 중...")
        mapped = 0
        unmapped = 0

        cursor.execute("SELECT id, name FROM recipe_ingredients")
        all_ri = cursor.fetchall()

        for row in all_ri:
            norm = normalized.get(row['name'])
            if norm and norm in name_to_id:
                cursor.execute(
                    "UPDATE recipe_ingredients SET ingredient_id = %s WHERE id = %s",
                    (name_to_id[norm], row['id'])
                )
                mapped += 1
            else:
                unmapped += 1

        conn.commit()

        # FK 체크 복구
        cursor.execute("SET FOREIGN_KEY_CHECKS = 1")
        conn.commit()

        log.info(f"재매핑 완료 - 매핑: {mapped}개 / 미매핑: {unmapped}개")

        # ── 결과 확인
        cursor.execute("SELECT COUNT(*) as cnt FROM ingredients")
        ing_count = cursor.fetchone()['cnt']

        cursor.execute("SELECT COUNT(*) as cnt FROM recipe_ingredients WHERE ingredient_id IS NOT NULL")
        mapped_count = cursor.fetchone()['cnt']

        cursor.execute("SELECT COUNT(*) as cnt FROM recipe_ingredients WHERE ingredient_id IS NULL")
        unmapped_count = cursor.fetchone()['cnt']

        log.info(f"=== 완료 ===")
        log.info(f"ingredients: {ing_count}개")
        log.info(f"recipe_ingredients 매핑됨: {mapped_count}개 / 미매핑: {unmapped_count}개")

        # ── 미매핑 샘플 출력 (확인용)
        cursor.execute("""
            SELECT DISTINCT name FROM recipe_ingredients 
            WHERE ingredient_id IS NULL 
            ORDER BY name 
            LIMIT 30
        """)
        unmapped_names = [row['name'] for row in cursor.fetchall()]
        log.info(f"미매핑 재료 샘플 (상위 30개): {unmapped_names}")

    except Error as e:
        log.error(f"DB 오류: {e}")
        conn.rollback()
    finally:
        cursor.close()
        conn.close()


if __name__ == "__main__":
    main()

2026-03-01 18:25:03,989 [INFO] === 재료 정규화 시작 ===
2026-03-01 18:25:04,001 [INFO] DB 연결 성공
2026-03-01 18:25:04,002 [INFO] 백업 테이블 생성 중...
2026-03-01 18:25:04,317 [INFO] 백업 완료 (ingredients_backup / recipe_ingredients_backup)
2026-03-01 18:25:04,323 [INFO] 고유 재료명 1107개 추출
2026-03-01 18:25:04,348 [INFO] 유효 재료: 707개 / 스킵: 133개
2026-03-01 18:25:04,348 [INFO] 스킵된 재료 샘플: ['꽃소금과 후춧가루', '파프리카 각 씩 청', '목록', '쌀뜨물', '정수물', '오징어 숙회 재료', '오징어 데침 재료', '초고추장 재료', '손질법', '물 과', '갈비찜재료', '데쳐서 만드는 법', '스테이크 재료', '스테이크소스 재료', '만능 양념장 재료', '닭갈비 재료', '족발덮밥 재료', '덮밥 소스 재료', '준비', '준비 법']
2026-03-01 18:25:04,349 [INFO] ingredients 테이블 재구축 중...
2026-03-01 18:25:05,030 [INFO] ingredients 테이블에 707개 재료 삽입 완료
2026-03-01 18:25:05,031 [INFO] recipe_ingredients 재매핑 중...
2026-03-01 18:25:06,320 [INFO] 재매핑 완료 - 매핑: 3501개 / 미매핑: 256개
2026-03-01 18:25:06,324 [INFO] === 완료 ===
2026-03-01 18:25:06,324 [INFO] ingredients: 707개
2026-03-01 18:25:06,325 [INFO] recipe_ingredients 매핑됨: 3501개 / 미매핑: 256개
2026-03-01 18:25:06,327 [INF